In [ ]:
import numpy as np
import anndata as an
import scanpy as sc
import pandas as pd
import os
import scipy
from scipy.io import mmread
import scipy.sparse as sp
import cell2location as c2l
import torch
import matplotlib

In [ ]:
counts = mmread('data/spatial/counts.mtx').T 
genes = pd.read_csv('data/spatial/genes.csv')['x'].values
barcodes = pd.read_csv('data/spatial/barcodes.csv')['x'].values

meta = pd.read_csv('data/spatial/meta.csv')
adata_st = an.AnnData(X=counts, obs=meta, var=pd.DataFrame(index=genes))
adata_st.X = sp.csr_matrix(adata_st.X)
adata_st 

In [ ]:
url = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/locus_types/gene_with_protein_product.txt"
hgnc = pd.read_csv(url, sep="\t")
pc_genes = hgnc["symbol"].dropna().unique().tolist()

keep = adata_st.var_names.intersection(pc_genes)
adata_st = adata_st[:, keep].copy()
adata_st

In [ ]:
adata_st.var["MT_gene"] = [
    gene.startswith("MT-") for gene in adata_st.var_names
]

adata_st.obsm["MT"] = adata_st[:, adata_st.var["MT_gene"].values].X.toarray()
adata_st= adata_st[:, ~adata_st.var["MT_gene"].values]
adata_st

In [ ]:
coor = pd.read_csv('data/spatial/coordinates.csv')
coor = coor.set_index('Unnamed: 0')
coor = coor.loc[adata_st.obs['barcode']]
coor

In [ ]:
adata_st.obsm['spatial'] = coor[['row', 'col']].values
adata_st

In [ ]:
adata_sc = sc.read_h5ad('data/spatial/scRNA.h5ad')
adata_sc.obs['assay'] = '10x'
adata_sc

In [ ]:
adata_sc.X = adata_sc.raw.X
shared_features = [
    feature for feature in adata_st.var_names if feature in adata_sc.var_names
]
adata_sc = adata_sc[:, shared_features]
adata_st = adata_st[:, shared_features]

In [ ]:
sc1 = adata_sc[adata_sc.obs['donor_id'] == 'BCLL-8-T']
st1 = adata_st[adata_st.obs['donor_id'] == 'BCLL-8-T']

In [ ]:
inf_aver = pd.read_csv("data/spatial/Cell2loc_output/donor_8/inf_aver.csv")

In [ ]:
model = c2l.models.Cell2location.load(
    'data/spatial/Cell2loc_output/donor_8',
    st1
)
model.view_anndata_setup()

In [ ]:
model.plot_history()

In [ ]:
st1 = model.export_posterior(
    st1,
    sample_kwargs={
        "num_samples": 1000,
        "batch_size": model.adata.n_obs
    },
)

In [ ]:
st1.uns["mod"]["factor_names"] = ['NBC_MBC','GCBC','PC_','CD4_T','Cytotoxic','myeloid','FDC','PDC','epithelial','preBC','preTC']
st1.obs[st1.uns["mod"]["factor_names"]] = st1.obsm[
    "q05_cell_abundance_w_sf"
]

In [ ]:
#st1.uns["mod"]["factor_names"] = ['NBC_MBC','GCBC','PC','CD4_T','Cytotoxic','myeloid','FDC','PDC','epithelial','preBC','preTC','']

In [ ]:
#slide = c2l.utils.select_slide(st1,"control_P1")

with matplotlib.rc_context({"figure.figsize": [4.5, 5]}):
    sc.pl.spatial(
        st1,
        cmap="magma",
        color=st1.uns["mod"]["factor_names"],
        ncols=4,
        size=1.3,
        img_key="hires",
        # limit color scale at 99.2% quantile of cell abundance
        vmin=0,
        spot_size=1,
        vmax="p99.2",
    )

In [ ]:
X = st1.obsm["q05_cell_abundance_w_sf"].to_numpy()
names = np.array(st1.uns["mod"]["factor_names"])

X_df = pd.DataFrame(
    X,
    index=st1.obs_names,  # spot barcodes
    columns=st1.obsm["q05_cell_abundance_w_sf"].columns  # cell types
)

X_df.to_csv("cell_abundance.csv")

st1.obs["dominant_celltype"] = names[X.argmax(axis=1)]
st1.obs["dominant_celltype"] = st1.obs["dominant_celltype"].astype("category")


sc.pl.spatial(
    st1,
    color="dominant_celltype",
    img_key="hires",
    spot_size=1,
)